In [1]:
import sys

!{sys.executable} -m pip install -q openai pandas scikit-learn rouge-score


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip


In [2]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from rouge_score import rouge_scorer

from openai import OpenAI

In [3]:
client = OpenAI(
    api_key="gsk_YOUR_GROQ_API_KEY",
    base_url="https://api.groq.com/openai/v1"
)

In [4]:
email_templates = [

{
"incoming":"I received the wrong product.",
"reply":"We're sorry for the inconvenience. We have initiated a replacement.",
"category":"Replacement"
},

{
"incoming":"Can I get a refund?",
"reply":"Your refund will be processed within five business days.",
"category":"Refund"
},

{
"incoming":"My account is locked.",
"reply":"Please reset your password using the Forgot Password option.",
"category":"Support"
},

{
"incoming":"My package has not arrived.",
"reply":"We're checking your shipment and will update you shortly.",
"category":"Shipping"
},

{
"incoming":"Can I change my delivery address?",
"reply":"Yes, if the order has not been shipped.",
"category":"Shipping"
}

]

dataset=[]

for i in range(100):
    dataset.extend(email_templates)

df=pd.DataFrame(dataset)

In [5]:
vectorizer = TfidfVectorizer()

email_vectors = vectorizer.fit_transform(df["incoming"])

In [6]:
def retrieve_examples(email):

    query = vectorizer.transform([email])

    similarity = cosine_similarity(query, email_vectors)[0]

    top = similarity.argsort()[::-1][:3]

    return df.iloc[top]

In [7]:
def generate_reply(email):

    examples = retrieve_examples(email)

    context=""

    for _,row in examples.iterrows():

        context += f"""

Incoming:
{row['incoming']}

Reply:
{row['reply']}

"""

    prompt=f"""
You are a professional customer support assistant.

Use these examples.

{context}

Incoming Email:

{email}

Write only the reply.
"""

    response = client.chat.completions.create(

        model="llama-3.3-70b-versatile",

        messages=[
            {
                "role":"user",
                "content":prompt
            }
        ],

        temperature=0.3

    )

    return response.choices[0].message.content

In [8]:
from rouge_score import rouge_scorer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def evaluate(reference, generated):

    # ROUGE-L Score
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    rouge = scorer.score(reference, generated)["rougeL"].fmeasure

    # Semantic Similarity
    tfidf = TfidfVectorizer()

    vectors = tfidf.fit_transform([reference, generated])

    semantic = cosine_similarity(vectors[0], vectors[1])[0][0]

    overall = (semantic * 0.7) + (rouge * 0.3)

    return {
        "Semantic Similarity": round(float(semantic * 100), 2),
        "ROUGE-L": round(float(rouge * 100), 2),
        "Overall": round(float(overall * 100), 2)
    }

In [9]:
incoming_email = """
Hello,

I received a damaged laptop today.

Can I replace it?

Thanks.
"""

# Generate reply
reply = generate_reply(incoming_email)

print("Generated Reply:\n")
print(reply)

# Reference (Ground Truth)
reference = """
Dear Customer,

We're sorry to hear that you received a damaged laptop.

We have initiated the replacement process.

Please share your order ID and photographs of the damaged laptop so we can verify the issue.

Once verified, your replacement laptop will be shipped immediately.

We sincerely apologize for the inconvenience.

Best Regards,
Customer Support Team
"""

# Evaluate
scores = evaluate(reference, reply)

print("\nEvaluation Scores:")
print(scores)

Generated Reply:

We're sorry for the inconvenience. We have initiated a replacement.

Evaluation Scores:
{'Semantic Similarity': 48.32, 'ROUGE-L': 20.9, 'Overall': 40.1}


In [10]:
results = pd.DataFrame({
    "Incoming Email": [incoming_email],
    "Generated Reply": [reply],
    "Reference Reply": [reference],
    "Semantic Similarity": [scores["Semantic Similarity"]],
    "ROUGE-L": [scores["ROUGE-L"]],
    "Overall Score": [scores["Overall"]]
})

results.to_csv("results.csv", index=False)

results

,Incoming Email,Generated Reply,Reference Reply,Semantic Similarity,ROUGE-L,Overall Score
0,"\nHello,\n\nI received a damaged laptop today....",We're sorry for the inconvenience. We have ini...,"\nDear Customer,\n\nWe're sorry to hear that y...",48.32,20.9,40.1


In [11]:
def evaluate(reference,generated):

    scorer=rouge_scorer.RougeScorer(["rougeL"])

    rouge=scorer.score(reference,generated)["rougeL"].fmeasure

    tfidf=TfidfVectorizer()

    matrix=tfidf.fit_transform([reference,generated])

    semantic=cosine_similarity(matrix[0],matrix[1])[0][0]

    overall=0.7*semantic+0.3*rouge

    return{

        "Semantic Similarity":round(semantic*100,2),

        "ROUGE-L":round(rouge*100,2),

        "Overall":round(overall*100,2)

    }

In [12]:
reference="""
We're sorry for the inconvenience.

We've started the replacement process.

You'll receive a new laptop shortly.
"""

scores=evaluate(reference,reply)

scores

{'Semantic Similarity': np.float64(53.09),
 'ROUGE-L': 53.33,
 'Overall': np.float64(53.16)}